# D2.2 · When the actor is an agent

**Function D — The Agentic SOC → The Agentic SOC — Response**  ·  *Security of AI*

Builds on **[D2.1 · Agent-assisted reconstruction](https://spbreed.github.io/cyber-commons/lessons/D2.1.html)**.

| | |
|---|---|
| Tools used | Keycloak, OpenSearch |

## What this lesson is

**What it covers.** Attribute an incident through the A2 `act` chain.

**Why a security engineer needs it.** "Which user" is now the wrong first question. The control it builds is: attribute to agent, authority, delegation chain and prompt.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

The internal actor was autonomous. Was it instructed, was it compromised, or did it simply do what it was allowed to do? None of your existing playbooks have a branch for that question, and the answer changes everything downstream.

> **At CyberTravels.** The internal actor was the Workflow Agent. Was it instructed, injected, or simply permitted? CyberTravels' existing playbook has no branch for that question, and every step of it assumes a person.

## 2 · The framework

```
   the internal actor was autonomous. which branch?

   instructed      someone told it to        -> who, and through what channel
   injected        content told it to        -> which corpus, written by whom
   permitted       it was allowed to         -> a control gap, not an intrusion

   no existing playbook has this branch, and it changes everything after it
```

Three responder instincts are correct for human incidents and misfire when the
actor is an agent.

1. **Disable the account.** For a human this stops them. For an agent holding an
   already-issued bearer token, it may not — the token remains valid until it
   expires.
2. **Interview the user.** They were asleep. They authorised a task; a model
   chose the actions. They cannot tell you what happened.
3. **Assume one actor.** There were three, in a chain, and only the last one
   touched the resource.

The correct first action is to **revoke the agent identity**, which is only
possible if A2 was done. This lesson is where the identity track's value becomes
operational rather than architectural.

<table style="border-collapse:collapse;margin:4px 0 2px;width:100%"><thead><tr><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">the runbook you have</th><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">the runbook this incident needs</th></tr></thead><tbody><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">1. disable the user account</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600">1. identify the &lt;b&gt;acting&lt;/b&gt; identity from the act chain (A2.5)</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">2. interview the user</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600">2. revoke that identity — no approval needed for a non-human (A3.6)</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">3. review the user&#x27;s recent activity</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600">3. scope by walking the delegation chain, not the host list (D2.3)</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px"></td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600">4. preserve the run trace before anything restarts (D2.5)</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px"></td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600">5. only then consider the human&#x27;s account, and say why</td></tr></tbody></table><div style="font-size:12px;color:#8A93A6;margin-top:6px">Every step on the left is correct for a human actor and wrong here. The human authorised a task; the actions were chosen by a model.</div>

## 3 · The procedure, as a skill

Disabling the human's account leaves both agents acting on tokens already issued. The skill enumerates the live sessions, simulates the reflex containment step, and separates the task the user authorised from the actions taken under it.

### The skill — [`skills/response/agent-actor-containment/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/response/agent-actor-containment/SKILL.md)

```yaml
name: agent-actor-containment
description: >-
  Establish what disabling a human's account does not stop when agents hold
  issued tokens, and separate the task a user authorised from the actions taken
  under it. Use during an incident where the actor is an agent and the account
  is a person's.
allowed-tools: Read, Grep, Glob
```

# Disabling her account stops her, not them

The reflex containment step is to disable the account the logs name. When the
actor is an agent holding an already-issued token, that step changes nothing:
the token is valid until it expires, and the agent keeps acting. Containment has
to name the credential and the workload, not the person.

## When to use this

Any incident where an agent acted on a user's behalf, and before any containment
decision that starts by disabling an account.

## Procedure

**1 — Enumerate live sessions and issued tokens per actor.** Human sessions,
agent sessions, and the tokens each holds with their expiry. This list is the
containment surface and it is usually longer than expected.

**2 — Simulate disabling the human's account.** Record what stops and what does
not. Already-issued tokens continuing to work is the finding, and it needs to be
stated before the containment call is made.

**3 — Interview to separate authorisation from action.** The user authorised a
*task*. Which of the actions taken under it did they know about, ask for, or
see? The answer is usually "the first one", and it changes the incident's
character entirely.

**4 — Draw the actor chain.** User, orchestrator, worker agent, downstream. The
logs show one actor; the chain shows three. Name each and what each can still
do.

**5 — Choose levers by what they actually stop.** Account disable, token
revocation, workload termination, downstream block. Record the effect and the
collateral of each, and pick from that table rather than from habit.

## Example

**Input** — the fixture committed at the top of [`scripts/agent_actor_containment.py`](scripts/agent_actor_containment.py). Edit it and re-run: the buckets, counts and verdicts below are derived from it, not hard-coded.

**Output** — the opening lines of a real run:

```
INSTINCT 1 — disable dana@corp's account
   dana@corp (human)      can act: True   account disabled, but the issued token is still valid
   patch-agent            can act: True   active
   deploy-agent           can act: True   active
   → the agents were never using her account interactively; they hold
     their own issued tokens, and one of them is acting AS her.

INSTINCT 2 — interview the user
```

The run continues past this. The script is the example: `test_skills.py` executes it on every build, so this block cannot drift from what the skill actually prints.

## Output contract

```json
{
  "sessions": [{"actor": "str", "kind": "human|agent", "tokens": [{"id": "str", "expires_in_s": 0}]}],
  "disable_human": {"stops": ["str"], "does_not_stop": ["str"]},
  "interview": {"authorised": "str", "aware_of": ["str"], "unaware_of": ["str"]},
  "chain": ["str"],
  "levers": [{"lever": "str", "stops": ["str"], "collateral": ["str"]}]
}
```

## Failure modes

- **Starting with the account.** It is the one lever that does not touch the
  actor.
- **Recording the user as having authorised the actions.** They authorised a
  task.
- **Containing the worker and not the orchestrator.** It will start another.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/response/agent-actor-containment/scripts/agent_actor_containment.py
SCRIPT = "skills/response/agent-actor-containment/scripts/agent_actor_containment.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; sparse-checkout then materialises only the two directories a
    # lesson needs: the procedures, and the repository they are run against.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    # `skills` is the procedures; `cybertravels` is the sample repository they
    # scan. Both, or the scanning skills clone successfully and then find
    # nothing to look at.
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set",
                    "skills", "cybertravels"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

Disabling the human's account leaves both agents able to act on already-issued tokens. The interview establishes the user authorised a task, not the actions. The chain shows three actors where the logs show one. Revoking `patch-agent`'s identity stops it in 12 seconds while dana and `deploy-agent` continue working.

## Your turn

Write your agentic incident runbook's first three steps. If step one is "disable the user account", rewrite it — and check whether you can currently revoke a single agent identity at all.

---

**Next → [D2.3 · Scoping an agentic incident](https://spbreed.github.io/cyber-commons/lessons/D2.3.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/D2.2.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/D2.2.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*